# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sohila-Khaled-Abbas/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook frames the **Refresh / Content Opportunity Scoring** lane into a formal ML task specification: task type, observed target, success metrics, unit of analysis dataframe probe, and justification over fixed rules.

## 1. My lane as an ML task (type)

**Selected Task Type:** **Ranking / Scoring** (Opportunity Priority Scoring)

**Why:** The primary decision requirement is to present content editors with a ranked queue of content items (ordered from highest opportunity / highest decay severity to lowest). Editors work from top to bottom. Ranking and continuous scoring reflect this operational decision workflow directly, unlike unranked binary classification.

In [1]:
# Task Type Definition
task_type = "Ranking / Priority Scoring"
output_format = "Ranked Queue of Content IDs (content_id)"
print(f"Task Type: {task_type}")
print(f"Primary Output: {output_format}")


Task Type: Ranking / Priority Scoring
Primary Output: Ranked Queue of Content IDs (content_id)


## 2. Target or proxy

- **Target:** Observed outcome measured in a future/subsequent performance window — e.g. observed traffic drop magnitude (`clicks_prev_30d` - `clicks_last_30d`) or post-refresh traffic recovery in panel data.
- **Label Origin:** Strictly **observed in data**, NEVER defined by an arbitrary heuristic rule.
- **Leakage Safeguard:** `is_declining_label`, `trend_direction`, and `trend_pct` are derived label targets or window summaries — they are strictly **excluded** from the feature set to prevent data leakage.

In [2]:
# Target & Feature Isolation Verification
forbidden_leakage_features = ["is_declining_label", "trend_direction", "trend_pct"]
print("Excluded Target/Leakage Features:", forbidden_leakage_features)
print("Rule: Features must only use observed metrics strictly prior to the outcome window.")


Excluded Target/Leakage Features: ['is_declining_label', 'trend_direction', 'trend_pct']
Rule: Features must only use observed metrics strictly prior to the outcome window.


## 3. Success metric

- **Primary Metric:** **Precision@K** (e.g. Precision@50 and Precision@100 on the top-ranked recommendations queue).
- **Secondary Metric:** **NDCG@K** (Normalized Discounted Cumulative Gain) for ranking quality, and **ROC-AUC** / **PR-AUC** for binary outcome proxies.
- **Defending the Metric:** Precision@K directly measures the efficiency of editor time — if an editor inspects the top 50 recommended pages, Precision@50 tells us what fraction actually represented high-impact decaying content.

In [3]:
# Metric definition check
primary_metric = "Precision@K (K=50, K=100)"
secondary_metric = "NDCG@K / ROC-AUC"
print(f"Primary Success Metric: {primary_metric}")
print(f"Secondary Metric: {secondary_metric}")


Primary Success Metric: Precision@K (K=50, K=100)
Secondary Metric: NDCG@K / ROC-AUC


## 4. The unit of analysis, as a real dataframe

**Unit of Analysis:** One row = One pseudonymized content item (`content_id`) within a client context (`client_id`).

Loading the starter CSV slice to verify dataset grain and schema:

In [4]:
import pandas as pd
import os

csv_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(csv_path):
    csv_path = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)

# Verify Grain: content_id must be unique per row
row_count = len(df)
unique_content = df["content_id"].nunique()
is_unique = (row_count == unique_content)

print(f"Dataframe Shape: {df.shape}")
print(f"Total Rows: {row_count:,}")
print(f"Unique Content IDs: {unique_content:,}")
print(f"Grain Verification (1 row = 1 content_id): {'PASSED' if is_unique else 'FAILED'}")

# Display key feature slice
feature_cols = ["content_id", "client_id", "impressions_90d", "clicks_90d", "content_age_days", "ctr", "avg_position", "engagement_rate"]
print("\nSample Dataframe Slice (Unit of Analysis):")
print(df[feature_cols].head())


Dataframe Shape: (30000, 44)
Total Rows: 30,000
Unique Content IDs: 30,000
Grain Verification (1 row = 1 content_id): PASSED

Sample Dataframe Slice (Unit of Analysis):
             content_id          client_id  ...  avg_position  engagement_rate
0  content_304f48230142  client_f369cb89fc  ...          10.6             5.88
1  content_a1fb4e703a9e  client_4e07408562  ...          20.3             0.00
2  content_9aa793d4d895  client_7f2253d7e2  ...          36.5             0.00
3  content_331d6c4de07b  client_19581e27de  ...           6.2             1.28
4  content_d99b7a2d90ca  client_3fdba35f04  ...          44.0             0.00

[5 rows x 8 columns]


## 5. Why ML beats a fixed rule here

A static rule (e.g., `if content_age > 180 and trend == down`) is inadequate for content refresh decisions because:
1. **Non-linear Multi-Factor Signal Interactions:** Traffic decline involves coupled interactions between search volume, position tier, CTR relative to position benchmark, engagement rate, keyword intent, and content type.
2. **False Positive & High-Volume Blind Spots:** A static rule flags old pages regardless of traffic volume (wasting editor time on 10-impression pages), while missing newly decaying high-traffic pages that need immediate intervention.
3. **Dynamic Prioritization:** Machine learning models learn non-linear decision boundaries and weight relative decay against opportunity size, delivering an optimized, actionable queue.

In [5]:
# Code demonstration showing why plain rule fails
# Rule: age > 180 days AND down trend
simple_rule_flag = (df["content_age_days"] > 180) & (df["trend_direction"] == "down")
flagged_by_rule = df[simple_rule_flag]

low_traffic_flagged = len(flagged_by_rule[flagged_by_rule["impressions_90d"] < 100])
high_traffic_missed = len(df[(~simple_rule_flag) & (df["impressions_90d"] > 5000) & (df["trend_direction"] == "down")])

print(f"Simple Rule Flagged Pages: {len(flagged_by_rule):,} total")
print(f" - Low-Traffic Wasted Effort (Impressions < 100): {low_traffic_flagged:,} pages")
print(f" - High-Traffic Missed Opportunities (Impressions > 5,000 & Down): {high_traffic_missed:,} pages")
print("Conclusion: A simple if-statement suffers from high false-positives and missed high-value opportunities.")


Simple Rule Flagged Pages: 8,564 total
 - Low-Traffic Wasted Effort (Impressions < 100): 1,649 pages
 - High-Traffic Missed Opportunities (Impressions > 5,000 & Down): 1,504 pages
Conclusion: A simple if-statement suffers from high false-positives and missed high-value opportunities.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`